# IT8104 Assessment 2 — Stage 2: Preprocessing and critical findings

This revision covers stages 1–2. Later sections arrive in the subsequent pull requests. Supplied CSVs are required locally. Notebook outputs are intentionally cleared for publication.

工程2までのコードです。以降の工程は後続PRで確認してください。

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# Embed plots in the saved notebook, as well as exporting PNG files.
get_ipython().run_line_magic("matplotlib", "inline")

### Load the datasets

Run this notebook with the working directory set to either the project folder or its `notebooks` folder. The original CSV files are loaded without modification. All `Sales` values in the supplied test file are empty, so it cannot directly provide supervised evaluation metrics. Task 2 therefore evaluates a holdout of labelled train.csv, then refits on eligible labelled rows to predict test.csv.

In [ ]:
project_root = next(
    (
        folder
        for folder in (Path.cwd(), Path.cwd().parent)
        if (folder / "data" / "train.csv").is_file()
        and (folder / "data" / "test.csv").is_file()
    ),
    None,
)
if project_root is None:
    raise FileNotFoundError(
        "Cannot find data/train.csv and data/test.csv. "
        "Open IT8104_Assessment 2 as the project folder, or run from its notebooks folder. "
        f"Current working directory: {Path.cwd()}"
    )

data_dir = project_root / "data"
train_df = pd.read_csv(data_dir / "train.csv")
test_df = pd.read_csv(data_dir / "test.csv")

### Initial data inspection

In [ ]:
print("Training dataset shape:", train_df.shape)
print("Test dataset shape:", test_df.shape)

In [ ]:
train_df.head()

In [ ]:
test_df.head()

In [ ]:
train_df.info()

In [ ]:
missing_summary = pd.DataFrame({
    "Missing Count": train_df.isna().sum(),
    "Missing Percentage": train_df.isna().mean() * 100,
})
missing_summary

In [ ]:
duplicate_count = train_df.duplicated().sum()
print("Duplicate rows:", duplicate_count)

### Descriptive statistics and category labels

These summaries use the original training dataset. `No` is a row sequence, so it is excluded from the substantive numeric summaries. Quantiles help distinguish the main distribution from extreme observations. No observations are removed in this section.

In [ ]:
from IPython.display import display

figure_dir = project_root / "outputs" / "figures"
table_dir = project_root / "outputs" / "tables"
figure_dir.mkdir(parents=True, exist_ok=True)
table_dir.mkdir(parents=True, exist_ok=True)

numeric_columns = ["Weight", "Visibility", "Price", "OperationYear", "Sales"]
numeric_summary = train_df[numeric_columns].describe(
    percentiles=[0.01, 0.25, 0.5, 0.75, 0.99]
).T
numeric_summary.to_csv(table_dir / "01_numeric_summary_raw.csv")
display(numeric_summary)

In [ ]:
categorical_columns = ["Fat", "Category", "Security", "StoreCode", "Size", "Area", "Type"]
category_tables = []
for column in categorical_columns:
    counts = train_df[column].fillna("<Missing>").value_counts(dropna=False)
    summary = counts.rename_axis("Label").reset_index(name="Count")
    summary["Percentage"] = summary["Count"] / len(train_df) * 100
    summary.insert(0, "Variable", column)
    category_tables.append(summary)
    print(column)
    display(summary)

categorical_summary = pd.concat(category_tables, ignore_index=True)
categorical_summary.to_csv(table_dir / "02_categorical_frequencies_raw.csv", index=False)

### Data quality checks

Negative weights and prices are inconsistent with their feature definitions. Negative sales require clarification: they could reflect errors or adjustments, and the README does not specify their meaning. Extreme positive values are review candidates, not automatically errors.

The README calls Visibility a percentage without specifying whether values use a 0–1 or 0–100 scale. Values above 100 exceed either interpretation; zero values need investigation. Opening years are also checked against the inspection year (2026), not an assumed dataset collection year. A rare historical year is not automatically invalid.

A repeated `ID`–`StoreCode` pair differs from a completely duplicated row. The dataset has no date field, so such pairs are recorded for review rather than automatically removed.

In [ ]:
quality_flags = pd.DataFrame({
    "Weight_nonpositive": train_df["Weight"].le(0),
    "Price_nonpositive": train_df["Price"].le(0),
    "Sales_negative_review": train_df["Sales"].lt(0),
    "Visibility_negative": train_df["Visibility"].lt(0),
    "Visibility_above_100": train_df["Visibility"].gt(100),
    "Opening_year_after_2026": train_df["OperationYear"].gt(2026),
})
quality_summary = quality_flags.sum().rename("Count").to_frame()
quality_summary["Percentage"] = quality_summary["Count"] / len(train_df) * 100
quality_summary.to_csv(table_dir / "03_quality_checks_raw.csv")
display(quality_summary)

suspect_records = train_df.loc[quality_flags.any(axis=1)].copy()
suspect_records["Review_reasons"] = quality_flags.loc[quality_flags.any(axis=1)].apply(
    lambda row: "; ".join(row.index[row]), axis=1
)
suspect_records.to_csv(table_dir / "04_records_for_review.csv", index=False)
display(suspect_records)

repeated_pair_rows = train_df.loc[train_df.duplicated(["ID", "StoreCode"], keep=False)]
repeated_pair_rows.to_csv(table_dir / "05_repeated_item_store_pairs.csv", index=False)
print("Completely duplicated rows:", train_df.duplicated().sum())
print("Repeated item-store pairs, beyond the first occurrence:",
      train_df.duplicated(["ID", "StoreCode"]).sum())
print("Rows involved in repeated item-store pairs:", len(repeated_pair_rows))
print("Zero visibility values:", train_df["Visibility"].eq(0).sum())
print("Opening year frequencies:")
display(train_df["OperationYear"].value_counts().sort_index())

### Initial inspection findings

- The training data contains 8,540 rows and 14 columns. Weight is missing in 1,466 rows (17.17%); Size is missing in 2,415 rows (28.28%). No complete row duplicates were found, but there are 15 repeated item–store pairs beyond their first occurrences.
- Weight ranges from −87,523 to 98,032; Price ranges from −13,479 to 35,127,361. Weight and Price each contain three negative values. Four Sales values are negative, while the maximum Sales value is 100,000,000. These observations need investigation before modelling.
- Visibility includes 526 zeros and an extreme value of 77,480,627. Its units and zero values need clarification. Opening years include 1876, 1957 and 3678; the last is later than the inspection year.
- Fat has five labels: Low Fat, LF, low fat, Regular and reg. A later preprocessing step can standardise the equivalent spellings after documenting the mapping.

The original values remain unchanged. Extreme values strongly affect means and standard deviations; medians and quantiles provide useful complementary descriptions.

## Univariate analysis

The following figures examine Sales, Price, Weight and Visibility, then the Category and Type frequencies. Each numeric figure shows the full raw histogram, the raw box plot and a focused histogram between the 1st and 99th percentiles. The box plot uses a symmetric log axis to show large positive values and any negative values together.

**The focused panel is a display choice, not a cleaning rule.** Its subtitle reports the number of non-missing values displayed and excluded. The complete data remains in `train_df`, and the raw panels include all non-missing observations. Quantiles are exploratory summaries, not thresholds to fit on future evaluation data.

In [ ]:
plt.rcParams.update({"figure.dpi": 110, "font.size": 10})
plot_focus_rows = []

for column in ["Sales", "Price", "Weight", "Visibility"]:
    values = train_df[column].dropna()
    lower, upper = values.quantile([0.01, 0.99])
    focused = values.loc[values.between(lower, upper)]
    outside_count = len(values) - len(focused)
    plot_focus_rows.append({
        "Variable": column, "Nonmissing": len(values),
        "Missing": train_df[column].isna().sum(),
        "Lower_1pct": lower, "Upper_99pct": upper,
        "Shown_in_focus": len(focused), "Outside_focus": outside_count,
    })

    fig, axes = plt.subplots(1, 3, figsize=(14, 4.2), layout="constrained")
    axes[0].hist(values, bins=40, color="#286A8D", edgecolor="white")
    axes[0].set_title(f"Raw histogram (n={len(values):,})")
    axes[0].set_ylabel("Record count")
    axes[0].ticklabel_format(axis="x", style="sci", scilimits=(0, 0))
    axes[1].boxplot(values, orientation="horizontal", showfliers=True,
                    flierprops={"markersize": 3, "alpha": 0.5})
    axes[1].set_xscale("symlog", linthresh=1)
    axes[1].xaxis.get_major_locator().set_params(numticks=7)
    axes[1].set_title("Raw box plot (symmetric log axis)")
    axes[1].set_yticks([])
    axes[2].hist(focused, bins=35, color="#25846E", edgecolor="white")
    axes[2].set_title(f"1st–99th percentile view\nShown {len(focused):,}; outside {outside_count:,}")
    axes[2].set_ylabel("Record count")
    for ax in axes:
        ax.set_xlabel(column + " (dataset units)")
        ax.grid(axis="y", alpha=0.2)
    fig.suptitle(f"{column}: raw distribution and focused view", fontsize=14)
    fig.savefig(figure_dir / f"univariate_{column.lower()}.png", dpi=160)
    display(fig)
    plt.close(fig)

plot_focus_summary = pd.DataFrame(plot_focus_rows)
plot_focus_summary.to_csv(table_dir / "06_numeric_plot_focus.csv", index=False)
display(plot_focus_summary)

In [ ]:
for column in ["Category", "Type"]:
    counts = train_df[column].fillna("<Missing>").value_counts().sort_values()
    fig, ax = plt.subplots(figsize=(10, max(3.8, len(counts) * 0.36)), layout="constrained")
    bars = ax.barh(counts.index, counts.values, color="#286A8D")
    labels = [f"{count:,} ({count / len(train_df):.1%})" for count in counts]
    ax.bar_label(bars, labels=labels, padding=4, fontsize=9)
    ax.set_xlim(0, counts.max() * 1.30)
    ax.set_xlabel("Number of training records")
    ax.set_title(f"{column}: record frequency (N={len(train_df):,})")
    ax.grid(axis="x", alpha=0.2)
    ax.set_axisbelow(True)
    fig.savefig(figure_dir / f"univariate_{column.lower()}.png", dpi=160)
    display(fig)
    plt.close(fig)

### Interpretation and transition to bivariate analysis

**Sales:** The raw mean is about 26,730.40, compared with a median of 1,794.33. The maximum (100,000,000) is far above the 99th percentile (7,412.54), so the mean does not describe the typical record well. The focused view is positively skewed, but investigation of extreme and negative targets must precede model fitting.

**Price:** The median is 143.03 and the central 98% lies approximately between 34.99 and 263.13. Negative prices and exceptionally large values are inconsistent with this main distribution. No currency is specified in the README, so these figures use dataset units.

**Weight:** The median non-missing weight is 12.60; the central 98% is approximately 5.15–20.85. Negative values and very large positive values warrant checking, while 17.17% missingness also needs a documented imputation strategy. The README does not give weight units.

**Visibility:** The median is approximately 0.054. The extreme observation compresses the raw histogram; the focused plot reveals the small values and mass at zero. Neither deleting zeros nor changing units is justified solely by their appearance.

**Category and Type:** Fruits and Vegetables (1,236 records; 14.47%) and Snack Foods (1,202; 14.07%) are the most common categories. Supermarket Type1 accounts for 5,588 records (65.43%). These are proportions of records, not revenue, demand or unique products, and they do not establish which groups have higher Sales.

**Transition:** The next section examines Price–Sales relationships and Sales by Category/Type, showing how suspicious observations affect the results. Then document preprocessing choices (label standardisation, missing values and anomalous values). Learned imputation, scaling and encoding should be fitted only on the modelling training split, not on the supplied test file or a held-out validation split. Repeated item–store pairs also need consideration when choosing a validation split.

For the later GA4 comparison, Category and Price provide conceptual links to product category and price. The retail dataset and Kiwi Outdoor are different sources; their observations cannot be joined as if they described the same transactions. Multivariate analysis and preprocessing follow below. The GA4 comparison is a separate assessment activity and is not performed here.

## Bivariate analysis: relationships with Sales

This section investigates associations with Sales, using the original training data. It does not remove or correct observations and does not use the supplied test file to estimate relationships.

For each numeric pair, Pearson correlation describes linear association; Spearman correlation describes rank association and is calculated by correlating average ranks after removing missing values for that pair. The table compares all available pairs with an exploratory **central window** retaining observations between each variable's own 1st and 99th percentiles. Pair-specific bounds and sample sizes are recorded.

The central window is a sensitivity check, not a validated clean dataset or a preprocessing rule. It excludes observations using Sales as well as the predictor, so its correlations describe a selected sample and can differ simply because the range is restricted. No p-values or causal effects are claimed. Box-plot zooms below only change axis limits; they retain the original full-data statistics.

In [ ]:
relationship_rows = []
relationship_bound_rows = []
relationship_windows = {}

for predictor in ["Price", "Weight", "Visibility", "OperationYear"]:
    pair = train_df[[predictor, "Sales"]].dropna()
    bounds = pair.quantile([0.01, 0.99])
    in_window = (
        pair[predictor].between(bounds.loc[0.01, predictor], bounds.loc[0.99, predictor])
        & pair["Sales"].between(bounds.loc[0.01, "Sales"], bounds.loc[0.99, "Sales"])
    )
    focused_pair = pair.loc[in_window]
    relationship_windows[predictor] = (pair, focused_pair)
    relationship_bound_rows.append({
        "Predictor": predictor,
        "Predictor_1pct": bounds.loc[0.01, predictor],
        "Predictor_99pct": bounds.loc[0.99, predictor],
        "Sales_1pct": bounds.loc[0.01, "Sales"],
        "Sales_99pct": bounds.loc[0.99, "Sales"],
        "Pairwise_missing": len(train_df) - len(pair),
        "Outside_central_window": len(pair) - len(focused_pair),
    })
    for view, frame in [("All available pairs", pair), ("Central window", focused_pair)]:
        relationship_rows.append({
            "Predictor": predictor, "View": view, "N": len(frame),
            "Pearson_r": frame.corr(method="pearson").loc[predictor, "Sales"],
            "Spearman_rho": frame.rank(method="average").corr().loc[predictor, "Sales"],
        })

relationship_summary = pd.DataFrame(relationship_rows)
relationship_bounds = pd.DataFrame(relationship_bound_rows)
relationship_summary.to_csv(table_dir / "07_sales_correlations.csv", index=False)
relationship_bounds.to_csv(table_dir / "08_relationship_window_bounds.csv", index=False)
display(relationship_summary.round(4))
display(relationship_bounds)

### Price and Sales

The left panel displays all Price–Sales pairs on symmetric log axes, keeping negative and extreme values visible. The right panel uses linear axes for the central window; colour indicates the number of records in each hexagonal bin. The panels have different scales and samples. Neither is a fitted prediction model.

In [ ]:
price_pair, price_focus = relationship_windows["Price"]
fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.8), layout="constrained")
axes[0].scatter(price_pair["Price"], price_pair["Sales"], s=12,
                color="#286A8D", alpha=0.25, edgecolors="none")
axes[0].set_xscale("symlog", linthresh=1)
axes[0].set_yscale("symlog", linthresh=1)
axes[0].xaxis.get_major_locator().set_params(numticks=7)
axes[0].yaxis.get_major_locator().set_params(numticks=7)
axes[0].set_title(f"All available pairs (n={len(price_pair):,})\nSymmetric log axes")
hexes = axes[1].hexbin(price_focus["Price"], price_focus["Sales"],
                       gridsize=38, mincnt=1, cmap="Blues", linewidths=0.1)
fig.colorbar(hexes, ax=axes[1], label="Records per bin")
axes[1].set_title(f"Central window (n={len(price_focus):,})\nLinear axes; {len(price_pair) - len(price_focus):,} pairs outside")
for ax in axes:
    ax.set_xlabel("Price (dataset units)")
    ax.set_ylabel("Sales (dataset units)")
    ax.grid(alpha=0.15)
fig.suptitle("Price and Sales: full-data and central-window views", fontsize=14)
fig.savefig(figure_dir / "bivariate_price_sales.png", dpi=160)
display(fig)
plt.close(fig)

### Sales by Category and Type

Group summaries report record counts, distinct items, distinct stores, mean, median, quartiles and extremes. The box plots on both sides use the same complete observations and quartiles; the right panel simply zooms to the global 1st–99th percentile range of Sales. Counts outside that viewport are included in the group tables and are not deleted.

Groups are ordered by median Sales. Store counts help interpret the Type comparison because many rows come from the same store. Rows and repeated item–store pairs should not be treated as independent experimental observations.

In [ ]:
sales_view_lower, sales_view_upper = train_df["Sales"].quantile([0.01, 0.99])
group_sales_summaries = {}

for group in ["Category", "Type"]:
    grouped = train_df.groupby(group, dropna=False)
    summary = grouped.agg(
        Records=("Sales", "size"), Sales_nonmissing=("Sales", "count"),
        Distinct_items=("ID", "nunique"), Distinct_stores=("StoreCode", "nunique"),
        Mean_Sales=("Sales", "mean"), Median_Sales=("Sales", "median"),
        Q1_Sales=("Sales", lambda values: values.quantile(0.25)),
        Q3_Sales=("Sales", lambda values: values.quantile(0.75)),
        Min_Sales=("Sales", "min"), Max_Sales=("Sales", "max"),
        Outside_zoom=("Sales", lambda values: (
            values.notna() & ~values.between(sales_view_lower, sales_view_upper)
        ).sum()),
    ).sort_values("Median_Sales")
    summary["Record_percentage"] = summary["Records"] / len(train_df) * 100
    group_sales_summaries[group] = summary
    number = "09" if group == "Category" else "10"
    summary.to_csv(table_dir / f"{number}_sales_by_{group.lower()}.csv")
    print(group)
    display(summary.round(2))

    labels = summary.index.tolist()
    values = [train_df.loc[train_df[group].eq(label), "Sales"].dropna() for label in labels]
    plot_labels = [f"{label} (n={summary.loc[label, 'Records']:,})" for label in labels]
    height = max(4.4, len(labels) * 0.42 + 1.6)
    fig, axes = plt.subplots(1, 2, figsize=(14, height), sharey=True, layout="constrained")
    for ax in axes:
        ax.boxplot(values, orientation="horizontal", tick_labels=plot_labels,
                   showfliers=True, flierprops={"markersize": 2.5, "alpha": 0.35},
                   medianprops={"color": "#C55B22", "linewidth": 1.7})
        ax.set_xlabel("Sales (dataset units)")
        ax.grid(axis="x", alpha=0.2)
    axes[0].set_xscale("symlog", linthresh=1)
    axes[0].xaxis.get_major_locator().set_params(numticks=7)
    axes[0].set_title("All raw values: symmetric log axis")
    axes[1].set_xlim(sales_view_lower, sales_view_upper)
    axes[1].set_title(f"Same statistics, zoomed axis\n{sales_view_lower:,.1f} to {sales_view_upper:,.1f}; tails outside view")
    fig.suptitle(f"Sales by {group}: ordered by median", fontsize=14)
    fig.savefig(figure_dir / f"bivariate_{group.lower()}_sales.png", dpi=160)
    display(fig)
    plt.close(fig)

### How extreme targets affect group averages

The next figure compares the raw mean and median on a log axis. It uses every available Sales value in each group. A large gap warns against describing a typical record using the mean alone; it does not prove that the large values are errors. The three largest Sales records are also listed for investigation. Their contribution is calculated against the sum of all raw Sales values, including negative values as supplied.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 7.5), layout="constrained")
for ax, group in zip(axes, ["Category", "Type"]):
    summary = group_sales_summaries[group]
    positions = np.arange(len(summary))
    ax.hlines(positions, summary["Median_Sales"], summary["Mean_Sales"], color="#BAC6CE")
    ax.scatter(summary["Mean_Sales"], positions, color="#C55B22", marker="D", s=35, label="Mean")
    ax.scatter(summary["Median_Sales"], positions, color="#286A8D", s=35, label="Median")
    ax.set_yticks(positions, summary.index)
    ax.set_xscale("log")
    ax.set_xlabel("Sales (dataset units, log axis)")
    ax.set_title(group + ": raw mean versus median")
    ax.grid(axis="x", alpha=0.2)
    ax.legend(loc="best")
fig.suptitle("Extreme observations can change comparisons of group averages", fontsize=14)
fig.savefig(figure_dir / "bivariate_group_mean_median.png", dpi=160)
display(fig)
plt.close(fig)

largest_sales_records = train_df.nlargest(3, "Sales")[
    ["No", "ID", "StoreCode", "Category", "Type", "Price", "Sales"]
]
raw_sales_sum = train_df["Sales"].sum()
largest_three_sum = largest_sales_records["Sales"].sum()
extreme_sales_contribution = pd.DataFrame([{
    "Raw_records": len(train_df), "Largest_records": len(largest_sales_records),
    "Raw_Sales_sum": raw_sales_sum, "Largest_three_Sales_sum": largest_three_sum,
    "Largest_three_share_percent": largest_three_sum / raw_sales_sum * 100,
}])
extreme_sales_contribution.to_csv(table_dir / "11_extreme_sales_contribution.csv", index=False)
largest_sales_records.to_csv(table_dir / "12_largest_sales_records.csv", index=False)
display(extreme_sales_contribution.round(2))
display(largest_sales_records)

### Bivariate findings and limits

- **Price–Sales:** On 8,540 available pairs, Pearson r is approximately 0.661 and Spearman rho is 0.563. In the central window (8,213 pairs), they are approximately 0.549 and 0.535. A positive association is visible in both views, but its magnitude depends on extreme values and sample selection. Higher Price does not by itself cause higher Sales; store type, category and the definition of Sales may contribute. No quantity-sold field is supplied, so this is not a demand or price-elasticity estimate.
- **Other numeric relationships:** Weight has a raw Pearson correlation of about 0.271 with Sales, but a rank correlation of only 0.015; the central-window Pearson correlation is about 0.021. This discrepancy shows why the raw linear correlation needs careful interpretation. Visibility has weak negative rank association (about −0.115); a near-zero raw Pearson value alone does not describe the complete pattern. Opening year shows little unadjusted association. These are descriptive comparisons, not feature-selection rules.
- **Category:** Seafood has the highest median Sales (2,055.32), but only 64 records. Baking Goods has the highest raw mean (106,130.03), while its median is only 1,581.94. A single exceptionally large target contributes to the mean. Category frequency, mean Sales and typical Sales answer different questions; none establishes profitability or overall customer demand.
- **Type:** Median Sales is highest for Supermarket Type3 (3,364.95), compared with Supermarket Type1 (1,991.41), Type2 (1,655.18), and Grocery Store (256.67). Type1 instead has the highest raw mean. Type3 and Type2 each represent only one distinct store in these data, so their differences cannot be separated from those particular stores by this comparison.
- **Extreme values:** Only three records account for approximately 91.86% of the raw Sales sum (including negative values as supplied). Their Category, Type, Price and row identifiers are retained for checking. They have not been discarded, replaced or accepted as verified sales.

**Business interpretation:** Price, Category and store context are useful candidates to investigate jointly. A single-variable comparison can mix their effects; it does not establish an actionable pricing policy or a causal advantage of a store format. For the later Kiwi Outdoor/GA4 discussion, product price, category, revenue and the view-to-purchase funnel are related concepts, but they come from a different dataset and require their own denominators and limits.

**Next step:** Add multivariate exploration (including category/type composition), then document and implement justified preprocessing. Resolve anomalous records and repeated item–store pairs before final model evaluation; fit learned imputation, encoding and scaling only on the training part of the chosen modelling split. The empty test targets still prevent direct calculation of test MAE, RMSE or R². The remaining Task 1 analysis and the Task 2 evaluation are implemented in the following sections.

## Task 1 — Preprocessing decisions and evidence

The raw sections above deliberately show the supplied data, including questionable records. The following rules create an **analysis copy**, with original values retained in CSVs and an audit trail. These are explicit working assumptions, not externally verified corrections.

| Issue | Treatment and rationale | Limitation |
|---|---|---|
| Sales < 0, missing/nonfinite, or ≥ 1,000,000 | Quarantine from the primary labelled analysis. Negative values could be returns, but no return semantics are given. The very large positives are separated by several orders of magnitude and include conflicting item–store records. | These targets are unverified, not proven errors. Scores apply to the retained population. Threshold sensitivity and rejected rows are shown. |
| Duplicate content except No | Remove only exact copies of every other field, keeping the first record. | Complete-row duplication is otherwise hidden by the unique sequence number. |
| Other repeated ID–StoreCode pairs | Keep differing observations and keep each pair in a single split/fold. | No date or transaction key exists to resolve conflicts. Do not arbitrarily keep the preferred target. |
| Weight ≤ 0 or > 100; Price ≤ 0 or > 1,000 | Mark as missing using fixed review bounds, rather than invent a replacement from an anomalous value. | Units are undocumented. These generous bounds are analyst assumptions supported by the distributional gap. |
| Visibility outside [0, 1] | Mark as missing. Retain zero. | Fractional storage is a working interpretation of the dominant observed range; the README calls this a percentage without a scale. No causal meaning is assigned to zeros. |
| OperationYear | Invalid years outside [1900, 2026] become missing. For a store with ≥5 valid observations and ≥90% agreement, use its training mode as the opening year, including correction of in-range minority years. | Opening year should be a store attribute; historical years are not rejected solely for being old. Strong store consistency, not the cutoff alone, supports repair. |
| Weight missing/flagged | First use the median valid Weight for the same item ID from the fitted subset; then a training-wide median for unseen/all-missing items. | Assumes weight is stable within an item. ID is used for this lookup only, not encoded as a regressor. |
| Fat | LF/low fat → Low Fat; reg → Regular. | A deterministic spelling map, with no target information. |
| Size missing | Keep an explicit Unknown category. Other categorical gaps also become Unknown. | Missingness is concentrated within stores. Filling a mode would imply unobserved store size. |
| Other numeric gaps | Median imputation. StandardScaler for numeric predictors and one-hot encoding for categoricals. | Learned on the fitted training subset only in Task 2. Tree models do not require scaling, but receive the same features for comparability. |

The EDA imputer below uses all eligible labelled rows **for descriptive Task 1 only**. Task 2 starts again from raw eligible rows and creates fresh unfitted pipeline instances after splitting. Nothing fitted in this EDA section enters evaluation. Full-data EDA informed analyst choices, so the final validation remains a development estimate rather than an untouched confirmatory test. This distinction follows the [scikit-learn guidance on data leakage](https://scikit-learn.org/stable/common_pitfalls.html#data-leakage).

In [ ]:
from IPython.display import Markdown
from sklearn.base import BaseEstimator, TransformerMixin, clone
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.utils.validation import check_is_fitted
import hashlib
import importlib.metadata
import json
import sys

RANDOM_STATE = 42
INSPECTION_YEAR = 2026  # A fixed inspection bound, not the unknown collection year.
TARGET_UPPER = 1_000_000
numeric_features = ["Weight", "Visibility", "Price", "OperationYear"]
categorical_features = ["Fat", "Category", "Security", "StoreCode", "Size", "Area", "Type"]
feature_columns = numeric_features + categorical_features
input_columns = ["ID"] + feature_columns
source_hashes = {name: hashlib.sha256((data_dir / name).read_bytes()).hexdigest()
                 for name in ["train.csv", "test.csv", "readme.txt"]}

def quality_masks(frame):
    return pd.DataFrame({
        "Weight_outside_review_bounds": frame["Weight"].notna() & ~frame["Weight"].between(0, 100, inclusive="right"),
        "Price_outside_review_bounds": frame["Price"].notna() & ~frame["Price"].between(0, 1000, inclusive="right"),
        "Visibility_outside_fraction_bounds": frame["Visibility"].notna() & ~frame["Visibility"].between(0, 1),
        "Year_outside_review_bounds": frame["OperationYear"].notna() & ~frame["OperationYear"].between(1900, INSPECTION_YEAR),
        "Target_quarantined": ~np.isfinite(frame["Sales"]) | frame["Sales"].lt(0) | frame["Sales"].ge(TARGET_UPPER),
    }, index=frame.index)

def eligible_rows(raw):
    target_bad = quality_masks(raw)["Target_quarantined"]
    duplicate = raw.duplicated(subset=[c for c in raw.columns if c != "No"], keep="first")
    audit = raw.copy()
    audit["Target_quarantined"] = target_bad
    audit["Duplicate_content_except_No"] = duplicate
    audit["Included"] = ~target_bad & ~duplicate
    return raw.loc[audit["Included"]].copy(), audit

class RetailFeatureCleaner(TransformerMixin, BaseEstimator):
    """Fixed rules plus item/store lookups learned only by fit(X). No target used."""
    def __init__(self, inspection_year=2026):
        self.inspection_year = inspection_year

    def _fixed(self, X):
        out = X.copy()
        for col, low, high in [("Weight", 0, 100), ("Price", 0, 1000),
                               ("Visibility", 0, 1), ("OperationYear", 1900, self.inspection_year)]:
            out[col] = pd.to_numeric(out[col], errors="coerce").astype(float)
            valid = out[col].between(low, high)
            if col in ["Weight", "Price"]:
                valid &= out[col].gt(low)
            out.loc[~valid, col] = np.nan
        for col in categorical_features:
            out[col] = out[col].astype(object)
            out.loc[out[col].isna(), col] = "Unknown"
        out["Fat"] = out["Fat"].replace({"LF": "Low Fat", "low fat": "Low Fat", "reg": "Regular"})
        return out

    def fit(self, X, y=None):
        fixed = self._fixed(X)
        self.item_weight_ = fixed.groupby("ID")["Weight"].median()
        self.store_year_ = {}
        for store, rows in fixed.groupby("StoreCode"):
            years = rows["OperationYear"].dropna()
            counts = years.value_counts()
            if len(years) >= 5 and counts.iloc[0] / len(years) >= .9:
                self.store_year_[store] = float(counts.index[0])
        self.fit_row_count_ = len(X)
        return self

    def transform(self, X):
        check_is_fitted(self, "item_weight_")
        out = self._fixed(X)
        out["Weight"] = out["Weight"].fillna(out["ID"].map(self.item_weight_))
        store_year = out["StoreCode"].map(self.store_year_)
        out["OperationYear"] = out["OperationYear"].where(store_year.isna(), store_year)
        return out[feature_columns]

def make_preprocessor():
    return Pipeline([
        ("rules", RetailFeatureCleaner(inspection_year=INSPECTION_YEAR)),
        ("columns", ColumnTransformer([
            ("num", Pipeline([("imputer", SimpleImputer(strategy="median")),
                               ("scaler", StandardScaler())]), numeric_features),
            ("cat", Pipeline([("imputer", SimpleImputer(strategy="constant", fill_value="Unknown")),
                               ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False,
                                                         drop="first"))]), categorical_features),
        ])),
    ])

eligible_df, row_audit = eligible_rows(train_df)
flags = quality_masks(train_df)
quality_rule_counts = flags.sum().rename("Rows").to_frame()
quality_rule_counts.to_csv(table_dir / "13_preprocessing_rule_counts.csv")
row_audit.to_csv(table_dir / "14_row_eligibility_audit.csv", index=False)
review_audit = train_df.loc[flags.any(axis=1)].join(flags.add_prefix("Flag_"))
review_audit.to_csv(table_dir / "15_preprocessing_review_records.csv", index=False)
display(quality_rule_counts)
display(pd.DataFrame({"Stage": ["Raw", "Target quarantine", "Exact duplicate (except No)", "Eligible"],
                      "Rows": [len(train_df), row_audit.Target_quarantined.sum(),
                               row_audit.Duplicate_content_except_No.sum(), len(eligible_df)]}))
display(review_audit)

# The review cutoffs are not fitted from a validation or test subset.
threshold_sensitivity = pd.DataFrame([
    {"Sales_upper_review_bound": cap,
     "Targets_quarantined": int((train_df.Sales.lt(0) | train_df.Sales.ge(cap) | ~np.isfinite(train_df.Sales)).sum())}
    for cap in [100_000, 1_000_000, 10_000_000]
])
threshold_sensitivity.to_csv(table_dir / "16_target_threshold_sensitivity.csv", index=False)
display(threshold_sensitivity)

In [ ]:
eda_cleaner = RetailFeatureCleaner().fit(eligible_df[input_columns])
eda_features = eda_cleaner.transform(eligible_df[input_columns])
eda_medians = eda_features[numeric_features].median()
clean_df = eligible_df[["No", "ID"]].join(eda_features).join(eligible_df[["Sales"]])
clean_df[numeric_features] = clean_df[numeric_features].fillna(eda_medians)
clean_df = clean_df[train_df.columns]
clean_df.to_csv(project_root / "outputs/cleaned_train_task1.csv", index=False)

# Record every changed feature, including missing-value imputation.
change_parts = []
for col in feature_columns:
    old, new = eligible_df[col], clean_df[col]
    same = old.eq(new).fillna(False) | (old.isna() & new.isna())
    idx = old.index[~same]
    if len(idx):
        change_parts.append(pd.DataFrame({"No": eligible_df.loc[idx, "No"], "Feature": col,
                                         "Original": old.loc[idx].astype(str),
                                         "Analysis_value": new.loc[idx].astype(str)}))
feature_changes = pd.concat(change_parts, ignore_index=True)
feature_changes.to_csv(table_dir / "17_task1_feature_changes.csv", index=False)
display(feature_changes.groupby("Feature").size().rename("Changed_cells").to_frame())
display(clean_df[numeric_features + ["Sales"]].describe().T)
display(Markdown(f"""### Recalculated cleaning summary
The CSV contains **{len(train_df):,}** labelled records and **{len(test_df):,}** test records.
All **{test_df.Sales.isna().sum():,}** test Sales entries are missing.
There are **{train_df.duplicated().sum()}** completely identical rows, but
**{row_audit.Duplicate_content_except_No.sum()}** exact duplicate content record(s) after ignoring No.
Quarantining **{row_audit.Target_quarantined.sum()}** unverified targets and removing duplicate content leaves
**{len(eligible_df):,}** eligible rows. Other conflicting pairs are retained and grouped for evaluation.
The EDA copy has **{clean_df.isna().sum().sum()}** missing cells after the documented transformations.
This does not establish that every source value is correct.

The previous chat's broad 1900–2026 rule alone would miss the in-range year 1957. Store-consistency checks address that value without claiming that 1957 is intrinsically impossible. A lower bound of 1900 also does not itself prove that 1876 is an error. Both are investigated against their store's dominant year.
"""))

## Task 1 — Bivariate analysis after preprocessing

These results use the explicitly defined analysis population, rather than the raw central-window view above. The change of population and values explains why these summaries need not equal the earlier raw summaries or previous chat estimates. Imputation can also attenuate or distort associations.

In [ ]:
clean_correlations = clean_df[numeric_features + ["Sales"]].corr()
clean_correlations.to_csv(table_dir / "18_clean_numeric_correlations.csv")
display(clean_correlations.round(4))
clean_group_summaries = {}
for col in categorical_features:
    summary = clean_df.groupby(col).agg(
        Count=("Sales", "size"), Mean_Sales=("Sales", "mean"),
        Median_Sales=("Sales", "median"), Distinct_stores=("StoreCode", "nunique"),
    ).sort_values("Mean_Sales", ascending=False)
    clean_group_summaries[col] = summary
    summary.to_csv(table_dir / f"19_clean_sales_by_{col.lower()}.csv")
    print(col)
    display(summary.round(2))

fig, axes = plt.subplots(2, 2, figsize=(15, 12), layout="constrained")
for ax, col in zip(axes.flat, ["Type", "Size", "Area", "Security"]):
    summary = clean_group_summaries[col].sort_values("Mean_Sales")
    ax.barh(summary.index, summary["Mean_Sales"], color="#286A8D")
    ax.set_title(f"Mean Sales by {col} (eligible EDA copy)")
    ax.set_xlabel("Sales (dataset units per record)")
    ax.grid(axis="x", alpha=.2)
fig.savefig(figure_dir / "clean_bivariate_store_features.png", dpi=160)
display(fig)
plt.close(fig)

quartile_df = clean_df.assign(Price_quartile=pd.qcut(clean_df.Price, 4, labels=["Q1 low", "Q2", "Q3", "Q4 high"]))
price_quartiles = quartile_df.groupby("Price_quartile", observed=True).Sales.agg(["count", "mean", "median"])
price_quartiles.to_csv(table_dir / "20_price_quartile_sales.csv")
display(price_quartiles)

## Task 1 — Multivariate analysis

The numeric heatmap considers several variables together. Cross-tabulations and store-level profiles investigate whether Size, Area and Type offer independent comparisons. The Type–Size heatmap reports both mean Sales and record counts, and the Price–Sales panels condition on store format. These are descriptive adjustments rather than causal controls. Empty combinations are shown as unavailable, not as zero Sales. Groups with only one store cannot support a general claim about that store format.

In [ ]:
store_profile = clean_df.groupby("StoreCode").agg(
    Records=("Sales", "size"), Mean_Sales=("Sales", "mean"),
    Type=("Type", "first"), Size=("Size", "first"), Area=("Area", "first"),
    Opening_year=("OperationYear", "first"),
)
store_profile.to_csv(table_dir / "21_store_profiles.csv")
display(store_profile)
for other in ["Size", "Area"]:
    counts = pd.crosstab(clean_df["Type"], clean_df[other])
    counts.to_csv(table_dir / f"22_type_by_{other.lower()}_counts.csv")
    display(counts)
security_mix = pd.crosstab(clean_df.Security, clean_df.Type, normalize="index")
security_mix.to_csv(table_dir / "23_security_type_proportions.csv")
display(security_mix.round(3))

fig, ax = plt.subplots(figsize=(7, 6), layout="constrained")
im = ax.imshow(clean_correlations, vmin=-1, vmax=1, cmap="RdBu_r")
ax.set_xticks(range(len(clean_correlations)), clean_correlations.columns, rotation=40, ha="right")
ax.set_yticks(range(len(clean_correlations)), clean_correlations.index)
for i in range(len(clean_correlations)):
    for j in range(len(clean_correlations)):
        v = clean_correlations.iloc[i, j]
        ax.text(j, i, f"{v:.2f}", ha="center", va="center", color="white" if abs(v) > .6 else "black")
ax.set_title("Numeric associations after documented preprocessing")
fig.colorbar(im, ax=ax, label="Pearson correlation")
fig.savefig(figure_dir / "multivariate_numeric_heatmap.png", dpi=160)
display(fig)
plt.close(fig)

mix_mean = clean_df.pivot_table(index="Type", columns="Size", values="Sales", aggfunc="mean")
mix_n = clean_df.pivot_table(index="Type", columns="Size", values="Sales", aggfunc="size").reindex_like(mix_mean)
mix_mean.to_csv(table_dir / "24_type_size_mean_sales.csv")
fig, ax = plt.subplots(figsize=(10, 5), layout="constrained")
im = ax.imshow(np.ma.masked_invalid(mix_mean.to_numpy()), cmap="Blues")
ax.set_xticks(range(len(mix_mean.columns)), mix_mean.columns)
ax.set_yticks(range(len(mix_mean.index)), mix_mean.index)
for i in range(len(mix_mean.index)):
    for j in range(len(mix_mean.columns)):
        v, n = mix_mean.iloc[i, j], mix_n.iloc[i, j]
        ax.text(j, i, "N/A" if pd.isna(v) else f"{v:,.0f}\n(n={int(n):,})", ha="center", va="center",
                color="white" if pd.notna(v) and v > .65 * np.nanmax(mix_mean) else "black")
fig.colorbar(im, ax=ax, label="Mean Sales (dataset units per record)")
ax.set_title("Sales by Type and Size: observed combinations")
fig.savefig(figure_dir / "multivariate_type_size_sales.png", dpi=160)
display(fig)
plt.close(fig)

fig, axes = plt.subplots(2, 2, figsize=(12, 9), sharex=True, sharey=True, layout="constrained")
within_type = []
for ax, (kind, rows) in zip(axes.flat, clean_df.groupby("Type")):
    hb = ax.hexbin(rows.Price, rows.Sales, gridsize=27, mincnt=1, cmap="Blues")
    r = rows[["Price", "Sales"]].corr().iloc[0, 1]
    nstores = rows.StoreCode.nunique()
    ax.set_title(f"{kind}: n={len(rows):,}, stores={nstores}, r={r:.3f}")
    ax.set_xlabel("Price (dataset units)")
    ax.set_ylabel("Sales (dataset units)")
    fig.colorbar(hb, ax=ax, label="Records per bin")
    within_type.append({"Type": kind, "Records": len(rows), "Stores": nstores, "Price_Sales_r": r})
within_type_df = pd.DataFrame(within_type)
within_type_df.to_csv(table_dir / "25_within_type_price_sales_correlations.csv", index=False)
fig.suptitle("Price and Sales within store formats")
fig.savefig(figure_dir / "multivariate_price_sales_by_type.png", dpi=160)
display(fig)
plt.close(fig)
display(within_type_df)

## Task 1 — Critical analysis and business findings

The following interpretation uses the recalculated tables, with no copied numerical estimates. Relationships are associations. In particular, store attributes that are constant or nearly constant within StoreCode cannot be interpreted as separable business effects.

In [ ]:
types = clean_group_summaries["Type"]
categories = clean_group_summaries["Category"]
security = clean_group_summaries["Security"]
price_r = clean_correlations.loc["Price", "Sales"]
type_store_counts = clean_df.groupby("Type").StoreCode.nunique()
display(Markdown(f"""
### Critical analysis
Price has a Pearson correlation of **{price_r:.3f}** with Sales in the analysis copy.
Within-format correlations range from **{within_type_df.Price_Sales_r.min():.3f}** to
**{within_type_df.Price_Sales_r.max():.3f}**. Persistence within format supports including Price as a predictor,
but does not establish that increasing a product's price would improve sales. The undefined Sales measure and absent quantity field prevent estimating demand elasticity or profit.

Mean Sales is **{types.loc['Supermarket Type3', 'Mean_Sales']:,.2f}** for Supermarket Type3 and
**{types.loc['Grocery Store', 'Mean_Sales']:,.2f}** for Grocery Store.
Type3 and Type2 each represent **{int(type_store_counts['Supermarket Type3'])}** and
**{int(type_store_counts['Supermarket Type2'])}** distinct store(s), respectively.
Size, Area, Type and opening year are linked to store identity in the profiles and cross-tabs.
The high mean for Medium stores therefore cannot be attributed to covered area alone.
No intervention on store size, format or location is identified by these observations.

Category means range from **{categories.Mean_Sales.min():,.2f}** to **{categories.Mean_Sales.max():,.2f}**.
Small categories have less stable summaries; the smallest contains **{int(categories.Count.min())}** records.
Security-provider means range from **{security.Mean_Sales.min():,.2f}** to **{security.Mean_Sales.max():,.2f}**.
The provider/type composition table helps check composition effects, but does not control for unobserved business differences.
These findings do not establish security-provider effectiveness or justify changing suppliers.

Weight–Sales Pearson r is **{clean_correlations.loc['Weight','Sales']:.3f}** and
Visibility–Sales r is **{clean_correlations.loc['Visibility','Sales']:.3f}**.
Weak marginal linear relationships do not rule out nonlinear effects or interactions.
Median imputation and repeated-item repair also influence those correlations.

### Business findings
1. **Plan by product and store context.** Price and store attributes justify comparing linear and nonlinear forecasting models. Their associations do not establish a pricing policy.
2. **Set differentiated targets.** The large descriptive gap across store formats suggests reviewing stock allocation and targets by store context, while recognising the small number of stores.
3. **Review product mix.** Mean Sales rises from **{price_quartiles.iloc[0]['mean']:,.2f}** in the lowest Price quartile to **{price_quartiles.iloc[-1]['mean']:,.2f}** in the highest. Category and price mix can guide investigation, but neither total demand nor margin can be inferred.
4. **Improve source data.** Validate the quarantined targets, inconsistent opening years and missing attributes with the data owner. Add sales period, units, returns definitions, quantities and promotions before operational forecasting.
5. **Use model output as planning support.** Unknown Size and zero Visibility are retained rather than interpreted as observed facts about physical capacity or exposure.

### Task 1 conclusion
Initial inspection, missing values, duplicate and category checks, raw and cleaned univariate/bivariate summaries,
multivariate visualisation, preprocessing rationale and business interpretation are now present.
The EDA export is descriptive. The next section reconstructs all fitted preprocessing within each training split.
"""))